# ScienceQA Visual Challenge: Starter Notebook

This notebook provides a starting point for the ScienceQA Visual Multiple-Choice Challenge. It is based on the provided baseline solution, but has been adapted to be more of a general-purpose starter.

**Objective:** Build a model that can answer visual multiple-choice questions based on scientific diagrams and text.

**Baseline Model:** `HuggingFaceTB/SmolVLM-500M-Instruct` (~500 M params)
**Fine-Tuning:** QLoRA (4-bit NF4)
**Scoring:** Multiple-choice log-likelihood

---

In [ ]:
# ── 0. Install libraries ──────────────────────────────────────────
# Run this cell to install the necessary Python packages.
# !pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes accelerate datasets pillow

In [ ]:
!pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes accelerate datasets pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 93.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 69.9 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
cd /content/drive/MyDrive/

/content/drive/MyDrive


In [ ]:
cd NYU/ECE-GY\ 7123\ DeepLearning

/content/drive/MyDrive/NYU/ECE-GY 7123 DeepLearning


In [ ]:
cd Final

/content/drive/MyDrive/NYU/ECE-GY 7123 DeepLearning/Final


In [ ]:
# ── 1. Imports & Configuration ───────────────────────────────────────────────
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Paths ────────────────────────────────────────────────────────────────────
# Adjust these paths to match your local environment
DATA_DIR   = Path("data")

# ── Model ────────────────────────────────────────────────────────────────────
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

# ── Basic Settings ───────────────────────────────────────────────────────────
IMG_SIZE        = 224

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


## 2. Load and Preprocess Data

In [ ]:
# ── 2a. Load CSVs ─────────────────────────────────────────────────────────────
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

# The 'choices' column is a JSON string, so we parse it
for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
train_df.head(2)

Train: 3,109 | Val: 1,048 | Test: 1,008


,id,image_path,question,choices,num_choices,answer,hint,lecture,solution,task,grade,subject,topic,category,skill
0,train_07667,images/train/train_07667.png,Why might putting each tadpole in its own pool...,[the male's tadpoles will be larger when they ...,3,2,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...
1,train_02628,images/train/train_02628.png,Why might forming strong social bonds with oth...,"[the female's offspring will live longer, the ...",3,0,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...


In [ ]:
# ── 2b. Improve the Prompt Format ────────────────────────────────────────────
CHOICE_LETTERS = "ABCDEFGHIJ"

def build_prompt(row: pd.Series, include_answer: bool = False) -> str:
    """
    Build a prompt for multiple-choice visual reasoning.
    """
    context_parts = []
    lecture = row.get("lecture", "")
    hint = row.get("hint", "")

    if pd.notna(lecture) and str(lecture).strip():
        context_parts.append(str(lecture).strip())
    if pd.notna(hint) and str(hint).strip():
        context_parts.append(str(hint).strip())

    context_str = "\n".join(context_parts)

    choices = row["choices"]
    choices_str = "\n".join(
        f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)
    )

    prompt = "<image>\n"
    prompt += "You are solving a multiple-choice science question.\n"
    prompt += "Choose the single best answer.\n"
    prompt += "Reply with only one capital letter.\n\n"

    if context_str:
        prompt += f"Context:\n{context_str}\n\n"

    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n\n"
    prompt += "Answer"

    if include_answer:
        answer_idx = int(row["answer"])
        prompt += f": {CHOICE_LETTERS[answer_idx]}"
    else:
        prompt += ":"

    return prompt

print(build_prompt(train_df.iloc[0], include_answer=True))

<image>
You are solving a multiple-choice science question.
Choose the single best answer.
Reply with only one capital letter.

Context:
Animals increase their reproductive success when they have offspring that survive to reproduce.
Animals can increase their chances of having offspring by behaving in ways that help them get partners to mate and reproduce with. These partners are called mates. For example, animals may make special sounds, perform specific dances, or show off bright colors to attract mates. Animals may also compete with each other for mates.
Animals can increase the chances that their offspring will survive to reproduce by caring for and protecting them. For example, animals may feed their offspring or guard them from predators. These behaviors increase the chances that the offspring will survive to adulthood, when they can reproduce.
Many behaviors can increase the chances that animals will have offspring that survive to reproduce. But the behaviors cannot guarantee th

In [ ]:
# ── 2c. PyTorch Dataset ───────────────────────────────────────────────────────
class ScienceQADataset(Dataset):
    def __init__(self, df: pd.DataFrame, data_dir: Path, img_size: int = 224, is_train: bool = True):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.img_size = img_size
        self.is_train = is_train

    def __len__(self) -> int:
        return len(self.df)

    def _load_image(self, rel_path: str) -> Image.Image:
        img = Image.open(self.data_dir / rel_path).convert("RGB")
        img = img.resize((self.img_size, self.img_size), Image.BICUBIC)
        return img

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]
        img = self._load_image(row["image_path"])

        if self.is_train:
            return {
                "image":  img,
                "text":   build_prompt(row, include_answer=True),
                "answer": int(row["answer"]),
            }
        else:
            return {
                "image":   img,
                "text":    build_prompt(row, include_answer=False),
                "choices": row["choices"],
                "answer":  int(row["answer"]) if "answer" in row else -1,
            }

train_ds = ScienceQADataset(train_df, DATA_DIR, img_size=IMG_SIZE, is_train=True)
val_ds   = ScienceQADataset(val_df,   DATA_DIR, img_size=IMG_SIZE, is_train=False)
test_ds  = ScienceQADataset(test_df,  DATA_DIR, img_size=IMG_SIZE, is_train=False)

print(f"Datasets created: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")

Datasets created: train=3109, val=1048, test=1008


## 3. Model Loading and Inference Example

This section loads `HuggingFaceTB/SmolVLM-500M-Instruct` and runs a quick inference example on one validation sample.

In [ ]:
# ── 3a. Load SmolVLM model + run one inference example ───────────────────────
from transformers import AutoProcessor, AutoModelForVision2Seq

processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True,
 )
if not torch.cuda.is_available():
    model.to(device)
model.eval()

# Pick a sample from validation set
sample = val_df.iloc[0]
sample_image = Image.open(DATA_DIR / sample["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
sample_prompt = build_prompt(sample, include_answer=False)

inputs = processor(
    text=[sample_prompt],
    images=[sample_image],
    return_tensors="pt",
)
inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

with torch.inference_mode():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
    )

decoded = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
print("Prompt:")
print(sample_prompt)
print("\nModel output:")
print(decoded)
print(f"\nGround-truth answer index: {sample['answer']}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Prompt:
<image>
You are solving a multiple-choice science question.
Choose the single best answer.
Reply with only one capital letter.

Context:
Animals increase their reproductive success when they have offspring that survive to reproduce.
Animals can increase their chances of having offspring by behaving in ways that help them get partners to mate and reproduce with. These partners are called mates. For example, animals may make special sounds, perform specific dances, or show off bright colors to attract mates. Animals may also compete with each other for mates.
Animals can increase the chances that their offspring will survive to reproduce by caring for and protecting them. For example, animals may feed their offspring or guard them from predators. These behaviors increase the chances that the offspring will survive to adulthood, when they can reproduce.
Many behaviors can increase the chances that animals will have offspring that survive to reproduce. But the behaviors cannot guar

In [ ]:
# ── 4a. Output Parsing Helpers ───────────────────────────────────────────────
import re

CHOICE_LETTERS = "ABCDEFGHIJ"

def decode_new_tokens(processor, generated_ids, inputs):
    prompt_len = inputs["input_ids"].shape[1]
    generated_only = generated_ids[:, prompt_len:]
    text = processor.batch_decode(generated_only, skip_special_tokens=True)[0].strip()
    return text

def extract_answer_index(text: str, num_choices: int) -> int:
    """
    Convert model outputs like:
    'A'
    'Answer: A'
    'A. the eggs will hatch'
    'A\n\nAnswer: E\n\nExplanation: ...'
    into a 0-indexed integer answer.
    """
    text_up = text.strip().upper()

    patterns = [
        r"ANSWER\s*[:\-]?\s*([A-J])",
        r"FINAL\s+ANSWER\s*[:\-]?\s*([A-J])",
        r"CORRECT\s+ANSWER\s*[:\-]?\s*([A-J])",
    ]

    for pattern in patterns:
        m = re.search(pattern, text_up)
        if m:
            idx = ord(m.group(1)) - ord("A")
            if 0 <= idx < num_choices:
                return idx

    lines = [line.strip() for line in text_up.splitlines() if line.strip()]

    for line in lines:
        m = re.fullmatch(r"([A-J])", line)
        if m:
            idx = ord(m.group(1)) - ord("A")
            if 0 <= idx < num_choices:
                return idx

        m = re.match(r"^([A-J])[\.\)\:\- ]", line)
        if m:
            idx = ord(m.group(1)) - ord("A")
            if 0 <= idx < num_choices:
                return idx

    m = re.search(r"\b([A-J])\b", text_up)
    if m:
        idx = ord(m.group(1)) - ord("A")
        if 0 <= idx < num_choices:
            return idx

    return 0

print("Helpers ready.")

Helpers ready.


In [ ]:
# ── 4b. Test One Validation Prediction ───────────────────────────────────────
sample = val_df.iloc[0]
sample_image = Image.open(DATA_DIR / sample["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
sample_prompt = build_prompt(sample, include_answer=False)

inputs = processor(
    text=[sample_prompt],
    images=[sample_image],
    return_tensors="pt",
)
inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

with torch.inference_mode():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False,
    )

decoded = decode_new_tokens(processor, generated_ids, inputs)
pred_idx = extract_answer_index(decoded, int(sample["num_choices"]))
true_idx = int(sample["answer"])

print("Raw model output:", repr(decoded))
print("Predicted index:", pred_idx)
print("Ground-truth index:", true_idx)
print("Correct:", pred_idx == true_idx)

Raw model output: "A. the leech's eggs will hatch."
Predicted index: 0
Ground-truth index: 0
Correct: True


In [ ]:
# ── 4c. Predict a Small Validation Subset ────────────────────────────────────
def predict_one_row(row):
    image = Image.open(DATA_DIR / row["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    prompt = build_prompt(row, include_answer=False)

    inputs = processor(
        text=[prompt],
        images=[image],
        return_tensors="pt",
    )
    inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=3,
            do_sample=False,
            temperature=None,
        )

    decoded = decode_new_tokens(processor, generated_ids, inputs)
    pred_idx = extract_answer_index(decoded, int(row["num_choices"]))
    return pred_idx, decoded

small_val = val_df.head(20).copy()

pred_indices = []
raw_outputs = []

for _, row in small_val.iterrows():
    pred_idx, decoded = predict_one_row(row)
    pred_indices.append(pred_idx)
    raw_outputs.append(decoded)

small_val["pred_answer"] = pred_indices
small_val["raw_output"] = raw_outputs
small_val["correct"] = small_val["pred_answer"] == small_val["answer"]

acc = small_val["correct"].mean()

print(f"Small validation accuracy: {acc:.4f}")
small_val[["id", "answer", "pred_answer", "correct", "raw_output"]].head(10)

Small validation accuracy: 0.3000


,id,answer,pred_answer,correct,raw_output
0,val_00671,0,0,True,A. the
1,val_04111,1,0,False,A. the
2,val_02022,3,0,False,A
3,val_01237,0,3,False,D
4,val_03458,4,4,True,E
5,val_04064,3,3,True,D
6,val_03959,4,0,False,A
7,val_03289,2,3,False,D
8,val_03452,3,4,False,E
9,val_01021,0,0,True,A


In [ ]:
# ── 5a. Run Validation Inference ─────────────────────────────────────────────
def predict_dataframe(df: pd.DataFrame, print_every: int = 50) -> pd.DataFrame:
    records = []

    for i, (_, row) in enumerate(df.iterrows(), start=1):
        pred_idx, decoded = predict_one_row(row)

        records.append({
            "id": row["id"],
            "pred_answer": pred_idx,
            "raw_output": decoded,
        })

        if i % print_every == 0 or i == len(df):
            print(f"Processed {i}/{len(df)}")

    return pd.DataFrame(records)

In [ ]:
# ── 5a. Run Validation Inference ─────────────────────────────────────────────
def predict_dataframe(df: pd.DataFrame, print_every: int = 50) -> pd.DataFrame:
    records = []

    for i, (_, row) in enumerate(df.iterrows(), start=1):
        pred_idx, decoded = predict_one_row(row)

        records.append({
            "id": row["id"],
            "pred_answer": pred_idx,
            "raw_output": decoded,
        })

        if i % print_every == 0 or i == len(df):
            print(f"Processed {i}/{len(df)}")

    return pd.DataFrame(records)

val_pred_df = predict_dataframe(val_df, print_every=50)

val_eval_df = val_df[["id", "answer"]].merge(val_pred_df, on="id", how="left")
val_eval_df["correct"] = val_eval_df["answer"] == val_eval_df["pred_answer"]

val_acc = val_eval_df["correct"].mean()

print(f"Validation accuracy: {val_acc:.4f}")
val_eval_df.head(10)

Processed 50/1048


KeyboardInterrupt: 

In [ ]:
# ── 5b. Generate Test Predictions ────────────────────────────────────────────
test_pred_df = predict_dataframe(test_df, print_every=50)

print(test_pred_df.shape)
test_pred_df.head(10)

Processed 50/1008
Processed 100/1008
Processed 150/1008
Processed 200/1008
Processed 250/1008
Processed 300/1008
Processed 350/1008
Processed 400/1008
Processed 450/1008
Processed 500/1008
Processed 550/1008
Processed 600/1008
Processed 650/1008
Processed 700/1008
Processed 750/1008
Processed 800/1008
Processed 850/1008
Processed 900/1008
Processed 950/1008
Processed 1000/1008
Processed 1008/1008
(1008, 3)


,id,pred_answer,raw_output
0,test_01750,1,B. The
1,test_00128,3,D
2,test_02891,0,A
3,test_02425,4,E
4,test_00930,4,E
5,test_03725,4,E
6,test_00009,0,A
7,test_02880,0,A. to
8,test_01208,1,B
9,test_00619,0,A) to


In [ ]:
# ── 5c. Save submission.csv ──────────────────────────────────────────────────
submission_df = test_pred_df[["id", "pred_answer"]].copy()
submission_df = submission_df.rename(columns={"pred_answer": "answer"})
submission_df["answer"] = submission_df["answer"].astype(int)

print(submission_df.head())
print(submission_df.shape)
print(submission_df.columns.tolist())

submission_df.to_csv("submission.csv", index=False)
print("Saved submission.csv")

           id  answer
0  test_01750       1
1  test_00128       3
2  test_02891       0
3  test_02425       4
4  test_00930       4
(1008, 2)
['id', 'answer']
Saved submission.csv


In [ ]:
# ── 5d. Check Submission Format ──────────────────────────────────────────────
check_df = pd.read_csv("submission.csv")

print(check_df.head())
print(check_df.shape)
print(check_df.columns.tolist())
print("Unique ids:", check_df["id"].nunique())
print("Total rows:", len(check_df))
print("Answer dtype:", check_df["answer"].dtype)
print("Answer min/max:", check_df["answer"].min(), check_df["answer"].max())

           id  answer
0  test_01750       1
1  test_00128       3
2  test_02891       0
3  test_02425       4
4  test_00930       4
(1008, 2)
['id', 'answer']
Unique ids: 1008
Total rows: 1008
Answer dtype: int64
Answer min/max: 0 4


--------------------------------------------------------

In [ ]:
# ── 5e. Score Choices Instead of Free Generation ─────────────────────────────
def score_candidate_answers(row):
    img_path = DATA_DIR / row["image_path"]

    if img_path.exists():
        image = Image.open(img_path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    else:
        print(f"Warning: missing image -> {img_path}")
        image = Image.new("RGB", (IMG_SIZE, IMG_SIZE), color="white")

    prompt = build_prompt(row, include_answer=False)

    prompt_inputs = processor(
        text=[prompt],
        images=[image],
        return_tensors="pt",
    )
    prompt_inputs = {
        k: v.to(model.device) if torch.is_tensor(v) else v
        for k, v in prompt_inputs.items()
    }

    prompt_len = prompt_inputs["input_ids"].shape[1]

    num_choices = int(row["num_choices"])
    letters = CHOICE_LETTERS[:num_choices]

    candidate_scores = []

    for letter in letters:
        full_text = prompt + f" {letter}"

        full_inputs = processor(
            text=[full_text],
            images=[image],
            return_tensors="pt",
        )
        full_inputs = {
            k: v.to(model.device) if torch.is_tensor(v) else v
            for k, v in full_inputs.items()
        }

        labels = full_inputs["input_ids"].clone()
        labels[:, :prompt_len] = -100

        with torch.inference_mode():
            outputs = model(**full_inputs, labels=labels)

        loss = outputs.loss.item()
        candidate_scores.append(loss)

    pred_idx = int(np.argmin(candidate_scores))
    pred_letter = letters[pred_idx]

    return pred_idx, pred_letter, candidate_scores

In [ ]:
# ── 5f. Test Choice Scoring on a Small Validation Subset ─────────────────────
small_val = val_df.head(20).copy()

pred_indices = []
pred_letters = []
score_lists = []

for _, row in small_val.iterrows():
    pred_idx, pred_letter, candidate_scores = score_candidate_answers(row)
    pred_indices.append(pred_idx)
    pred_letters.append(pred_letter)
    score_lists.append(candidate_scores)

small_val["pred_answer_scored"] = pred_indices
small_val["pred_letter_scored"] = pred_letters
small_val["score_list"] = score_lists
small_val["correct_scored"] = small_val["pred_answer_scored"] == small_val["answer"]

acc_scored = small_val["correct_scored"].mean()

print(f"Small validation accuracy (choice scoring): {acc_scored:.4f}")
small_val[["id", "answer", "pred_answer_scored", "pred_letter_scored", "correct_scored", "score_list"]].head(10)

Small validation accuracy (choice scoring): 0.3000


,id,answer,pred_answer_scored,pred_letter_scored,correct_scored,score_list
0,val_00671,0,0,A,True,"[0.8454309701919556, 0.8766809701919556, 2.751..."
1,val_04111,1,0,A,False,"[0.5394719243049622, 1.4769718647003174, 3.570..."
2,val_02022,3,0,A,False,"[0.280489444732666, 2.842989444732666, 2.34298..."
3,val_01237,0,3,D,False,"[1.9368849992752075, 2.077509880065918, 3.5228..."
4,val_03458,4,4,E,True,"[2.1462926864624023, 2.6697301864624023, 3.786..."
5,val_04064,3,3,D,True,"[1.932227611541748, 6.150977611541748, 6.79941..."
6,val_03959,4,0,A,False,"[0.8725167512893677, 2.833454132080078, 4.2006..."
7,val_03289,2,3,D,False,"[1.5489009618759155, 2.939526081085205, 3.7442..."
8,val_03452,3,4,E,False,"[2.0630078315734863, 3.3286328315734863, 3.852..."
9,val_01021,0,0,A,True,"[0.6843599081039429, 3.1140475273132324, 3.606..."


-------------------------------------------------------

In [ ]:
# ── 5g. Build a Hint-Only Prompt ─────────────────────────────────────────────
def build_prompt_hint_only(row: pd.Series, include_answer: bool = False) -> str:
    """
    Build a prompt that keeps only the hint and removes the lecture.
    """
    hint = row.get("hint", "")
    hint_str = ""

    if pd.notna(hint) and str(hint).strip():
        hint_str = str(hint).strip()

    choices = row["choices"]
    choices_str = "\n".join(
        f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)
    )

    prompt = "<image>\n"
    prompt += "You are solving a multiple-choice science question.\n"
    prompt += "Choose the single best answer.\n"
    prompt += "Reply with only one capital letter.\n\n"

    if hint_str:
        prompt += f"Hint:\n{hint_str}\n\n"

    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n\n"
    prompt += "Answer"

    if include_answer:
        answer_idx = int(row["answer"])
        prompt += f": {CHOICE_LETTERS[answer_idx]}"
    else:
        prompt += ":"

    return prompt

print(build_prompt_hint_only(train_df.iloc[0], include_answer=True))

<image>
You are solving a multiple-choice science question.
Choose the single best answer.
Reply with only one capital letter.

Hint:
Animals often behave in certain ways that can increase their reproductive success. Read the passage about a specific animal behavior. Then, follow the instructions below.

Amazonian poison frogs live in tropical forests in northern South America. After a male and female frog mate, the female frog lays eggs on a plant. When tadpoles hatch from the eggs, the male frog lets the tadpoles climb onto his back. The male then searches for water trapped in the spaces where plants' leaves meet their stems. He puts his tadpoles in these small pools of water.
If the male frog puts a tadpole into a pool with a larger tadpole, the smaller tadpole is often eaten. So, the male frog usually puts each tadpole into a pool of water that does not have other tadpoles in it. Each tadpole lives in its own pool until it undergoes metamorphosis to develop into a frog.
Figure: an 

In [ ]:
# ── 5h. Add a Prompt-Switchable Prediction Function ──────────────────────────
def predict_one_row_with_prompt(row, prompt_fn):
    img_path = DATA_DIR / row["image_path"]

    if img_path.exists():
        image = Image.open(img_path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    else:
        print(f"Warning: missing image -> {img_path}")
        image = Image.new("RGB", (IMG_SIZE, IMG_SIZE), color="white")

    prompt = prompt_fn(row, include_answer=False)

    inputs = processor(
        text=[prompt],
        images=[image],
        return_tensors="pt",
    )
    inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=3,
            do_sample=False,
            temperature=None,
        )

    decoded = decode_new_tokens(processor, generated_ids, inputs)
    pred_idx = extract_answer_index(decoded, int(row["num_choices"]))
    return pred_idx, decoded

print("Prompt-switchable predictor ready.")

Prompt-switchable predictor ready.


In [ ]:
# ── 5i. Compare Current Prompt vs Hint-Only on Validation Subset ────────────
val_subset = val_df.head(200).copy()

current_preds = []
hint_only_preds = []

for i, (_, row) in enumerate(val_subset.iterrows(), start=1):
    pred_current, _ = predict_one_row_with_prompt(row, build_prompt)
    pred_hint_only, _ = predict_one_row_with_prompt(row, build_prompt_hint_only)

    current_preds.append(pred_current)
    hint_only_preds.append(pred_hint_only)

    if i % 20 == 0 or i == len(val_subset):
        print(f"Processed {i}/{len(val_subset)}")

val_subset["pred_current"] = current_preds
val_subset["pred_hint_only"] = hint_only_preds

acc_current = (val_subset["pred_current"] == val_subset["answer"]).mean()
acc_hint_only = (val_subset["pred_hint_only"] == val_subset["answer"]).mean()

print(f"Current prompt accuracy on subset:   {acc_current:.4f}")
print(f"Hint-only prompt accuracy on subset: {acc_hint_only:.4f}")

Processed 20/200
Processed 40/200
Processed 60/200
Processed 80/200
Processed 100/200
Processed 120/200
Processed 140/200
Processed 160/200
Processed 180/200
Processed 200/200
Current prompt accuracy on subset:   0.6300
Hint-only prompt accuracy on subset: 0.5300


--------------------------------------------------------

In [ ]:
# ── 5j. Build a Minimal Prompt ───────────────────────────────────────────────
def build_prompt_minimal(row: pd.Series, include_answer: bool = False) -> str:
    """
    Build a prompt with only the image, question, and choices.
    """
    choices = row["choices"]
    choices_str = "\n".join(
        f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)
    )

    prompt = "<image>\n"
    prompt += "You are solving a multiple-choice science question.\n"
    prompt += "Choose the single best answer.\n"
    prompt += "Reply with only one capital letter.\n\n"
    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n\n"
    prompt += "Answer"

    if include_answer:
        answer_idx = int(row["answer"])
        prompt += f": {CHOICE_LETTERS[answer_idx]}"
    else:
        prompt += ":"

    return prompt

print(build_prompt_minimal(train_df.iloc[0], include_answer=True))

<image>
You are solving a multiple-choice science question.
Choose the single best answer.
Reply with only one capital letter.

Question: Why might putting each tadpole in its own pool of water increase the reproductive success of a male Amazonian poison frog? Complete the claim below that answers this question and is best supported by the passage.
Putting each tadpole in its own pool of water increases the chances that ().
Choices:
A. the male's tadpoles will be larger when they hatch
B. the male will carry his tadpoles through the forest
C. the male's tadpoles will become adult frogs

Answer: C


In [ ]:
# ── 5k. Compare Current Prompt vs Minimal Prompt ─────────────────────────────
val_subset = val_df.head(200).copy()

current_preds = []
minimal_preds = []

for i, (_, row) in enumerate(val_subset.iterrows(), start=1):
    pred_current, _ = predict_one_row_with_prompt(row, build_prompt)
    pred_minimal, _ = predict_one_row_with_prompt(row, build_prompt_minimal)

    current_preds.append(pred_current)
    minimal_preds.append(pred_minimal)

    if i % 20 == 0 or i == len(val_subset):
        print(f"Processed {i}/{len(val_subset)}")

val_subset["pred_current"] = current_preds
val_subset["pred_minimal"] = minimal_preds

acc_current = (val_subset["pred_current"] == val_subset["answer"]).mean()
acc_minimal = (val_subset["pred_minimal"] == val_subset["answer"]).mean()

print(f"Current prompt accuracy on subset:  {acc_current:.4f}")
print(f"Minimal prompt accuracy on subset:  {acc_minimal:.4f}")

Processed 20/200
Processed 40/200
Processed 60/200
Processed 80/200
Processed 100/200
Processed 120/200
Processed 140/200
Processed 160/200
Processed 180/200
Processed 200/200
Current prompt accuracy on subset:  0.6300
Minimal prompt accuracy on subset:  0.4050


--------------------------------------------------------

LoRA

In [ ]:
# ── 6a. Load PEFT and Attach LoRA ────────────────────────────────────────────
!pip -q install peft

from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True,
)

if not torch.cuda.is_available():
    model.to(device)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


trainable params: 2,080,768 || all params: 509,563,072 || trainable%: 0.4083


In [ ]:
# ── 6b. Build a Training Dataset for LoRA ────────────────────────────────────
class ScienceQATrainDataset(Dataset):
    def __init__(self, df: pd.DataFrame, data_dir: Path, img_size: int = 224):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.img_size = img_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.data_dir / row["image_path"]

        if img_path.exists():
            image = Image.open(img_path).convert("RGB").resize((self.img_size, self.img_size))
        else:
            image = Image.new("RGB", (self.img_size, self.img_size), color="white")

        prompt_text = build_prompt(row, include_answer=False)
        full_text = build_prompt(row, include_answer=True)

        return {
            "image": image,
            "prompt_text": prompt_text,
            "full_text": full_text,
        }

train_lora_ds = ScienceQATrainDataset(train_df, DATA_DIR, img_size=IMG_SIZE)
val_lora_ds = ScienceQATrainDataset(val_df, DATA_DIR, img_size=IMG_SIZE)

print(len(train_lora_ds), len(val_lora_ds))

3109 1048


In [ ]:
# ── 6c. Create the Collate Function ──────────────────────────────────────────
def lora_collate_fn(batch):
    images = [item["image"] for item in batch]
    prompt_texts = [item["prompt_text"] for item in batch]
    full_texts = [item["full_text"] for item in batch]

    prompt_inputs = processor(
        text=prompt_texts,
        images=images,
        return_tensors="pt",
        padding=True,
    )

    full_inputs = processor(
        text=full_texts,
        images=images,
        return_tensors="pt",
        padding=True,
    )

    labels = full_inputs["input_ids"].clone()

    for i in range(labels.size(0)):
        prompt_len = int((prompt_inputs["attention_mask"][i] == 1).sum().item())
        labels[i, :prompt_len] = -100

    if processor.tokenizer.pad_token_id is not None:
        labels[labels == processor.tokenizer.pad_token_id] = -100

    batch_out = {
        "input_ids": full_inputs["input_ids"],
        "attention_mask": full_inputs["attention_mask"],
        "pixel_values": full_inputs["pixel_values"],
        "labels": labels,
    }

    if "pixel_attention_mask" in full_inputs:
        batch_out["pixel_attention_mask"] = full_inputs["pixel_attention_mask"]

    return batch_out

In [ ]:
# ── 6d. Create Dataloaders ───────────────────────────────────────────────────
train_loader = DataLoader(
    train_lora_ds,
    batch_size=2,
    shuffle=True,
    collate_fn=lora_collate_fn,
)

val_loader = DataLoader(
    val_lora_ds,
    batch_size=2,
    shuffle=False,
    collate_fn=lora_collate_fn,
)

print("Dataloaders ready.")

Dataloaders ready.


In [ ]:
# ── 6e. Train for One Epoch ──────────────────────────────────────────────────
from torch.optim import AdamW
from tqdm.auto import tqdm

model.train()

optimizer = AdamW(model.parameters(), lr=1e-4)

train_losses = []

for step, batch in enumerate(tqdm(train_loader, desc="Training")):
    batch = {
        k: v.to(model.device) if torch.is_tensor(v) else v
        for k, v in batch.items()
    }

    outputs = model(**batch)
    loss = outputs.loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    train_losses.append(loss.item())

    if (step + 1) % 50 == 0:
        print(f"Step {step+1}: loss = {np.mean(train_losses[-50:]):.4f}")

print(f"Final training loss: {np.mean(train_losses):.4f}")

Training:   0%|          | 0/1555 [00:00<?, ?it/s]

Step 50: loss = 0.8167
Step 100: loss = 0.6808
Step 150: loss = 0.6116
Step 200: loss = 0.4909
Step 250: loss = 0.4799
Step 300: loss = 0.5772
Step 350: loss = 0.5545
Step 400: loss = 0.4849
Step 450: loss = 0.4982
Step 500: loss = 0.3615
Step 550: loss = 0.4601
Step 600: loss = 0.3690
Step 650: loss = 0.4130
Step 700: loss = 0.5035
Step 750: loss = 0.4062
Step 800: loss = 0.4583
Step 850: loss = 0.3241
Step 900: loss = 0.4151
Step 950: loss = 0.4372
Step 1000: loss = 0.4200
Step 1050: loss = 0.3908
Step 1100: loss = 0.4372
Step 1150: loss = 0.4429
Step 1200: loss = 0.3888
Step 1250: loss = 0.4028
Step 1300: loss = 0.4401
Step 1350: loss = 0.3899
Step 1400: loss = 0.6402
Step 1450: loss = 0.4978
Step 1500: loss = 0.4117
Step 1550: loss = 0.3834
Final training loss: 0.4707


In [ ]:
# ── 6f. Evaluate the LoRA Model on Validation ────────────────────────────────
model.eval()

val_pred_df_lora = predict_dataframe(val_df, print_every=50)

val_eval_df_lora = val_df[["id", "answer"]].merge(val_pred_df_lora, on="id", how="left")
val_eval_df_lora["correct"] = val_eval_df_lora["answer"] == val_eval_df_lora["pred_answer"]

val_acc_lora = val_eval_df_lora["correct"].mean()

print(f"LoRA validation accuracy: {val_acc_lora:.4f}")
val_eval_df_lora.head(10)

Processed 50/1048
Processed 100/1048
Processed 150/1048
Processed 200/1048
Processed 250/1048
Processed 300/1048
Processed 350/1048
Processed 400/1048
Processed 450/1048
Processed 500/1048
Processed 550/1048
Processed 600/1048
Processed 650/1048
Processed 700/1048
Processed 750/1048
Processed 800/1048
Processed 850/1048
Processed 900/1048
Processed 950/1048
Processed 1000/1048
Processed 1048/1048
LoRA validation accuracy: 0.7347


,id,answer,pred_answer,raw_output,correct
0,val_00671,0,1,B. the,False
1,val_04111,1,0,A. the,False
2,val_02022,3,0,A,False
3,val_01237,0,1,B,False
4,val_03458,4,1,B,False
5,val_04064,3,3,D,True
6,val_03959,4,3,D,False
7,val_03289,2,0,A,False
8,val_03452,3,3,D,True
9,val_01021,0,3,D,False


In [ ]:
# ── 6g. Generate Test Predictions with LoRA ──────────────────────────────────
model.eval()

test_pred_df_lora = predict_dataframe(test_df, print_every=50)

print(test_pred_df_lora.shape)
test_pred_df_lora.head(10)

Processed 50/1008
Processed 100/1008
Processed 150/1008
Processed 200/1008
Processed 250/1008
Processed 300/1008
Processed 350/1008
Processed 400/1008
Processed 450/1008
Processed 500/1008
Processed 550/1008
Processed 600/1008
Processed 650/1008
Processed 700/1008
Processed 750/1008
Processed 800/1008
Processed 850/1008
Processed 900/1008
Processed 950/1008
Processed 1000/1008
Processed 1008/1008
(1008, 3)


,id,pred_answer,raw_output
0,test_01750,1,B. The
1,test_00128,0,A
2,test_02891,0,A
3,test_02425,1,B
4,test_00930,1,B
5,test_03725,1,B
6,test_00009,1,B
7,test_02880,1,B
8,test_01208,1,B
9,test_00619,0,A


In [ ]:
# ── 6h. Save LoRA Submission ─────────────────────────────────────────────────
submission_df_lora = test_pred_df_lora[["id", "pred_answer"]].copy()
submission_df_lora = submission_df_lora.rename(columns={"pred_answer": "answer"})
submission_df_lora["answer"] = submission_df_lora["answer"].astype(int)

print(submission_df_lora.head())
print(submission_df_lora.shape)
print(submission_df_lora.columns.tolist())

submission_df_lora.to_csv("submission_lora.csv", index=False)
print("Saved submission_lora.csv")

           id  answer
0  test_01750       1
1  test_00128       0
2  test_02891       0
3  test_02425       1
4  test_00930       1
(1008, 2)
['id', 'answer']
Saved submission_lora.csv


In [ ]:
# ── 6i. Check LoRA Submission Format ─────────────────────────────────────────
check_df_lora = pd.read_csv("submission_lora.csv")

print(check_df_lora.head())
print(check_df_lora.shape)
print(check_df_lora.columns.tolist())
print("Unique ids:", check_df_lora["id"].nunique())
print("Total rows:", len(check_df_lora))
print("Answer dtype:", check_df_lora["answer"].dtype)
print("Answer min/max:", check_df_lora["answer"].min(), check_df_lora["answer"].max())

           id  answer
0  test_01750       1
1  test_00128       0
2  test_02891       0
3  test_02425       1
4  test_00930       1
(1008, 2)
['id', 'answer']
Unique ids: 1008
Total rows: 1008
Answer dtype: int64
Answer min/max: 0 3
